### Universidade Federal da Bahia
### Escola Politécnica
### Programa de Pós-Graduação em Engenharia Industrial
### Discente: Ghabriel Anton Gomes de Sá
### Orientadores: Marcelo Embiruçu e Cristiano Fontes

<center><h1>UMA ABORDAGEM COMBINANDO UMA ESTIMATIVA DE PESOS INICIAIS BASEADA EM LINEARIZAÇÃO E ALGORITMO CONSTRUTIVISTA PARA REDES NEURAIS DE PROPAGAÇÃO DIRETA COM UMA ÚNICA CAMADA OCULTA APLICADAS À IDENTIFIÇÃO DE MODELOS COM MÚLTIPLAS ENTRADAS E ÚNICA SAÍDA EM PROBLEMAS DE CLASSIFICAÇÃO BINÁRIA</center></h1>

#### <p style='text-align: justify;'>O Método para a obtenção dos pesos iniciais propõe que: (i) seja realizada a linearização, via série de Taylor, do modelo não-linear a ser utilizado para a modelagem do conjunto de dados (o modelo consiste em uma rede neural com p entradas, uma única camada oculta e um neurônio na camada de saída), onde o ponto de equilíbrio para a linearização é a média das suas respectivas variáveis (x,y); (ii) seja realizada a Regressão Linear Múltipla, via Método dos Mínimos Quadrados, do conjunto de dados a ser modelado; (iii) através da comparação matemática entre (i) e (ii), obtenha-se os pesos que conectam os neurônios de entrada ao primeiro neurônio oculto da rede. 
#### <p style='text-align: justify;'>Em seguida, a rede continua a aumentar o número de neurônios ocultos a partir de uma abordagem construtivista. Toda a rede é treinada a cada adição de um novo neurônio oculto, sendo que os pesos (e bias) obtidos no final de um treinamento são utilizados como pesos iniciais para o treinamento subsequente. Os pesos iniciais do neurônio oculto a ser adicionado são obtidos a partir do método de Bengio e Glorot (2010), conhecido como método de Xavier; biases são inicializados com valores nulos. Comparou-se o método proposto (WILCA-BC) com outros dois métodos (RIXM e ELM).

## Importação de pacotes

In [ ]:
# Manipulação de dados:
import pandas as pd
from sklearn import preprocessing

# Resolução de sistema de eq. não-lineares:
import statsmodels.api as sm
from scipy.optimize import fsolve

# Álgebra linear:
import numpy as np
import math
from scipy.linalg import pinv #Usado no ELM

# Cálculo da acurácia:
from sklearn.metrics import accuracy_score

# Contagem de tempo:
import time

# Criação e plotagem de gráficos:
import matplotlib.pyplot as plt
%matplotlib inline

# BCE:
import tensorflow as tf

## Fixação de seed
Garante a reprodutibilidade dos experimentos.

In [ ]:
np.random.seed(0)

## Importação do banco de dados
<p style='text-align: justify;'>Para que a importação do banco de dados seja feita da maneira correta, é necessário que o banco de dados a ser importado esteja de acordo com o padrão apresentado a seguir e também que ele esteja localizado na mesma pasta que esse código. A princípio, esse código está implementado em um arquivo .ipynb.</p>

O banco de dados deve apresentar, obrigatoriamente, as seguintes características:

-Deve ser um arquivo do tipo .csv;

-Os dados devem estar delimitados por ponto e vírgula (;), ou seja, devem estar presentes em diferentes células da planilha Excel;

-As variáveis de entradas são todas as colunas, exceto a última (que trata-se da variável de saída);

-Pode ou não possuir header.

In [ ]:
# Pedindo informações do conjunto de dados (nome do conjunto e se ele possui cabeçalho ou não):
filename = input("Digite o nome do arquivo .csv que contém o banco de dados:")
headdecider = input(
    "O conjunto de dados possui header (nome dos atributos)? Responda com S ou N.")

# Abrindo conjunto de dados:
if headdecider in ['S']:
    data = pd.read_csv(filename+'.csv', sep=';')
elif headdecider in ['N']:
    data = pd.read_csv(filename+'.csv', header=None, sep=';')
else:
    print("Você digitou um comando incorreto. Os comandos possíveis são 'S' ou 'N'")

## Pré-processamento de dados

#### • Obtendo primeiras informações sobre o conjunto de dados:

In [ ]:
# Mostrando cinco primeiras linhas de dados:
data.head()

In [ ]:
# Mostrando informações sobre os atributos:
data.info()

In [ ]:
# Descrição estatística dos dados:
data.describe()

#### • Lidando com os dados faltantes:

In [ ]:
# Verificando a existência de dados faltantes:
data.isna().sum()

In [ ]:
# Deletando dados faltantes:
data.dropna(inplace=True)

In [ ]:
# Missing values também podem estar representadas através do número 0. Por conta disso, deve-se verificar se os zeros são missing values ou se são
# inerentes ao atributo:
num_zeros = (data == 0).sum()
print('Quantidade de valores nulos:\n', num_zeros)

#### • Embaralhando os dados:

In [ ]:
from sklearn.utils import shuffle
data = shuffle(data, random_state=1)

#### • Reduzindo a escala dos dados:
Essa etapa garante que os dados estejam entre 0 e 1.

In [ ]:
# Cria uma array a partir dos dados:
z = data.values

# Atribuindo nova escala os dados:
min_max_scaler = preprocessing.MinMaxScaler()
data_normalized = min_max_scaler.fit_transform(z)

In [ ]:
print("Dados normalizados:\n", data_normalized)

#### • Delimitação de entrada e saída:

In [ ]:
# Delimitando:
inputs_all = data_normalized[:, :-1]
targets_all = data_normalized[:, -1]

# Atribuindo valor ao número de atributos (número de colunas dos dados de entrada):
n_inputs = inputs_all.shape[1]

In [ ]:
# Exibindo valores em tela:
print("Inputs:\n", inputs_all)
print("Outputs:\n", targets_all)
print("Nº de atributos:\n", n_inputs)

#### • Divisão do conjunto de dados em treinamento e teste

In [ ]:
# Definindo o número total de amostras:
samples_count = data_normalized.shape[0]


# Definindo o número de amostras em cada subconjunto, supondo que desejemos uma distribuição 80-20 de treinamento e teste:
train_samples_count = int(0.8 * samples_count)
test_samples_count = int(0.2 * samples_count)

# Criando variáveis que registram as entradas e saídas para treinamento:
train_inputs = inputs_all[:train_samples_count]
train_targets = targets_all[:train_samples_count]

# Criando variáveis que registram as entradas e targets para teste:
test_inputs = inputs_all[train_samples_count:]
test_targets = targets_all[train_samples_count:]

## Criação dos Modelos
<p style='text-align: justify;'>Os modelos são do tipo SFNN e treinados de acordo com o algoritmo de retropropagação (ARP); exceto o Método 3, que utiliza o ELM.</p>

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy()

## MÉTODO 1 - WILCA-BC
Modelo SFNN, treinado via ARP, considerando os pesos iniciais (e bias) do primeiro neurônio oculto obtidos a partir de Linearização e Análise via Série de Taylor e pesos dos demais neurônios ocultos, bem como do neurônio de saída obtidos a partir do método de Xavier, preservando os pesos (e bias) dos neurônios já aprendidos e utilizando-os como estimativas iniciais para o próximo treinamento.

### Aplicação de Regressão Linear no conjunto de dados
<p style='text-align: justify;'>A regressão é feita utilizando o módulo statsmodels.api. O método utilizado para a regressão é o Método dos Mínimos Quadrados.</p>

In [ ]:
# Adiciona a coluna das constantes
inputs_all_cte = sm.add_constant(inputs_all, prepend=True)
res = sm.OLS(targets_all, inputs_all_cte).fit()  # Cria e ajusta o modelo
print(res.summary())  # Imprime os resultados

### Resolução do sistema de equações proposto para obter os parâmetros iniciais da rede
<p style='text-align: justify;'>A resolução é feita utilizando o módulo scipy.optimize.fsolve. O módulo retorna as raízes das equações (não lineares) definidas por f(x)= 0, dada uma estimativa inicial.</p>

In [ ]:
# Definindo constantes:
x_M = []
for i in range(n_inputs):
    x_M.append(np.mean(inputs_all[:, i]))
k = []
for i in range(n_inputs):
    k.append(res.params[i+1])
b = res.params[0]

In [ ]:
# Definindo chutes iniciais para beta e wi como valores aleatórios entre 0 e 1:
x0 = np.random.rand(n_inputs+1, 1)

In [ ]:
# Definindo função sigmoide e sua derivada:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def dsigmoid(x):
    z = sigmoid(x)
    return z * (1 - z)

In [ ]:
# Inicializando variáveis wi e beta:
w = [0 for i in range(n_inputs)]
beta = 0

In [ ]:
# Definindo sistema de equações a ser resolvido:
def f(x):
    w, beta = x[:-1], x[-1]
    f = [sigmoid(np.dot(w, x_M)+beta)-(sigmoid(np.dot(w, x_M)+beta) -
                                       (sigmoid(np.dot(w, x_M)+beta))**2)*np.dot(w, x_M)-b]
    for i in range(n_inputs):
        f.append((sigmoid(np.dot(w, x_M)+beta) -
                 (sigmoid(np.dot(w, x_M)+beta))**2)*w[i]-k[i])
    return f

In [ ]:
# Resolvendo o sistema e imprimindo resultados (pesos e bias iniciais):
result = fsolve(f, x0)
print(result)

### Criando Método 1:

In [ ]:
class Method1:

    def __init__(self, x, y, n):
        # vetor de saídas preditas. Inicialmente um vetor de zeros.
        self.Yh = np.zeros((1, y.shape[0]))
        self.n = n  # número de neurônios ocultos.
        # número de neurônios em cada camada (entrada, intermediária e saída).
        self.dims = [x.shape[0], n, 1]
        # um dicionário que conterá os parâmetros W e b entre as camadas da rede (entrada-intermediária e intermediária-saída); ou seja: W1, b1, W2 e b2.
        self.param = {}
        self.ch = {}  # parâmetro intermediário de cálculo.
        self.grad = {}  # parâmetro intermediário de cálculo.
        self.loss = None  # erro de treinamento (BCE).
        self.lr = 0.1  # taxa de aprendizagem.
        # número de observações do conjunto de treinamento.
        self.sam = y.shape[0]

    def nInit(self):
        self.param['W1'] = np.random.uniform(-math.sqrt(6/(self.dims[0] + self.dims[1])), math.sqrt(
            6/(self.dims[0] + self.dims[1])), (self.dims[1], self.dims[0]))
        self.param['b1'] = np.zeros((self.dims[1], 1))
        self.param['W2'] = np.random.uniform(-math.sqrt(6/(self.dims[1] + self.dims[2])), math.sqrt(
            6/(self.dims[1] + self.dims[2])), (self.dims[2], self.dims[1]))
        self.param['b2'] = np.zeros((self.dims[2], 1))
        self.param['W1'][0] = result[:-1]
        self.param['b1'][0] = result[-1]
        return

    def forward(self, x):
        Z1 = self.param['W1'].dot(x) + self.param['b1']
        A1 = sigmoid(Z1)
        self.ch['Z1'], self.ch['A1'] = Z1, A1
        Z2 = self.param['W2'].dot(A1) + self.param['b2']
        A2 = sigmoid(Z2)
        self.ch['Z2'], self.ch['A2'] = Z2, A2
        self.Yh = A2
        return self.Yh

    def backward(self, x, y):
        dLoss_Yh = - (y-self.Yh[0])

        dLoss_Z2 = dLoss_Yh * dsigmoid(self.ch['Z2'])
        dLoss_A1 = np.dot(self.param["W2"].T, dLoss_Z2)
        dLoss_W2 = 1/self.ch['A1'].shape[1] * np.dot(dLoss_Z2, self.ch['A1'].T)
        dLoss_b2 = 1/self.ch['A1'].shape[1] * \
            np.dot(dLoss_Z2, np.ones([dLoss_Z2.shape[1], 1]))

        dLoss_Z1 = dLoss_A1 * dsigmoid(self.ch['Z1'])
        dLoss_A0 = np.dot(self.param["W1"].T, dLoss_Z1)
        dLoss_W1 = 1/x.shape[1] * np.dot(dLoss_Z1, x.T)
        dLoss_b1 = 1/x.shape[1] * \
            np.dot(dLoss_Z1, np.ones([dLoss_Z1.shape[1], 1]))

        self.param["W1"] = self.param["W1"] - self.lr * dLoss_W1
        self.param["b1"] = self.param["b1"] - self.lr * dLoss_b1
        self.param["W2"] = self.param["W2"] - self.lr * dLoss_W2
        self.param["b2"] = self.param["b2"] - self.lr * dLoss_b2

    def train(self, x, y, iter=1501):
        for i in range(0, iter):
            self.forward(x)
            self.backward(x,y)
        self.loss = bce(y, self.Yh[0])
        return

    def test(self, x, y):
        self.forward(x)
        return bce(y, self.Yh[0])

### Executando Método 1:

In [ ]:
# Inicializando contagem do tempo:
ini = time.time()

# Abrindo listas para BCE e Acurácia (treino e teste):
bce_train_1 = []
bce_test_1 = []
acc_train_1 = []
acc_test_1 = []

# Abrindo lista para pesos:
params_models = []

# Sinalização, em tela, início de rede com 1 neurônio oculto:
print("Iniciando rede com 1 neurônio oculto:")

# Definindo entrada e saída de treino:
x = train_inputs.T
y = train_targets.T

# Criando modelo:
Modelo = Method1(x, y, 1)

# Inicializando pesos:
Modelo.nInit()

# Treinando modelo:
Modelo.train(x, y)

# BCE de treino:
bce_train_1.append(Modelo.loss)
print("BCE de treino:", Modelo.loss)

# Acurácia de treino:
for n in range(0, Modelo.Yh.shape[1]):
    if Modelo.Yh[0][n] > 0.5:
        Modelo.Yh[0][n] = 1
    else:
        Modelo.Yh[0][n] = 0
acc_train_1.append(accuracy_score(y, Modelo.Yh[0]))
print("Acurácia de treino:", accuracy_score(y, Modelo.Yh[0]))

# Registrando pesos já aprendidos:
params_models.append(Modelo.param)

# Definindo entrada e saída de teste:
x = test_inputs.T
y = test_targets.T

# Testando modelo e calculando BCE de teste:
bce_test_1.append(Modelo.test(x, y))

# Acurácia de teste:
for n in range(0, Modelo.Yh.shape[1]):
    if Modelo.Yh[0][n] > 0.5:
        Modelo.Yh[0][n] = 1
    else:
        Modelo.Yh[0][n] = 0
Yh = [item for elem in Modelo.Yh for item in elem]
y = np.ndarray.tolist(test_targets)
acc_test_1.append(accuracy_score(y, Yh))
print("Acurácia de teste:", accuracy_score(y, Yh))

In [ ]:
# Inicializando loop para treinamento de rede com demais neurônios:
for n in range(2, 251):
    # Sinalização, em tela, de início de rede com n neurônios ocultos:
    print("Iniciando rede com", n, "neurônios ocultos:")

    # Definindo entrada e saída de treino:
    x = train_inputs.T
    y = train_targets.T

    # Criando Modelo:
    Modelo = Method1(x, y, n)

    # Inicializando novos pesos e aproveitando pesos do treinamento anterior:
    Modelo.nInit()
    Modelo.param['W1'][:-1] = params_models[-1]['W1']
    Modelo.param['b1'][:-1] = params_models[-1]['b1']
    Modelo.param['W2'][-1][:-1] = params_models[-1]['W2']
    Modelo.param['b2'] = params_models[-1]['b2']

    # Treinando modelo:
    Modelo.train(x, y)

    # Registro e exibição de BCE de treino:
    bce_train_1.append(Modelo.loss)
    print("BCE de treino:", Modelo.loss)

    # Registro e exibição de Acurácia de treino:
    for n in range(0, Modelo.Yh.shape[1]):
        if Modelo.Yh[0][n] > 0.5:
            Modelo.Yh[0][n] = 1
        else:
            Modelo.Yh[0][n] = 0
    acc_train_1.append(accuracy_score(y, Modelo.Yh[0]))
    print("Acurácia de treino:", accuracy_score(y, Modelo.Yh[0]))

    # Registrando pesos já aprendidos:
    params_models.append(Modelo.param)

    # Definindo entrada e saída de teste:
    x = test_inputs.T
    y = test_targets.T

    # Testando modelo e calculando BCE de teste:
    bce_test_1.append(Modelo.test(x, y))

    # Registro e exibição de Acurácia de teste:
    for n in range(0, Modelo.Yh.shape[1]):
        if Modelo.Yh[0][n] > 0.5:
            Modelo.Yh[0][n] = 1
        else:
            Modelo.Yh[0][n] = 0
    acc_test_1.append(accuracy_score(y, Modelo.Yh[0]))
    print("Acurácia de teste:", accuracy_score(y, Modelo.Yh[0]))

### ANÁLISE DA PERFORMANCE (MÉTRICAS: FUNÇÃO DE PERDA [BCE DE TREINO], ACURÁCIA DE TREINO, BCE DE TESTE E ACURÁCIA DE TESTE)

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento:", fim-ini)

In [ ]:
# Criação e exibição de Gráfico do BCE de treino:
plt.plot(bce_train_1)
plt.ylabel('Train BCE')
plt.xlabel('Hidden Nodes')
plt.legend(['WILCA-BC'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico da Acurácia de treino:
plt.plot(acc_train_1)
plt.ylabel('Train Accuracy')
plt.xlabel('Hidden Nodes')
plt.legend(['WILCA-BC'])
plt.show()

# Exibindo em tela Acurácia de treino associada ao melhor modelo gerado:
print("Acurácia de treino associada ao modelo com maior Acurácia de teste:",
      acc_train_1[acc_test_1.index(max(acc_test_1))])

In [ ]:
# Criação e exibição de Gráfico do BCE de teste:
plt.plot(bce_test_1)
plt.ylabel('Test BCE')
plt.xlabel('Hidden Nodes')
plt.legend(['WILCA-BC'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico da Acurácia de teste:
plt.plot(acc_test_1)
plt.ylabel('Test Accuracy')
plt.xlabel('Hidden Nodes')
plt.legend(['WILCA-BC'])
plt.show()

# Exibindo, em tela, Acurácia máxima de teste:
print("Acurácia máxima de teste obtido:", max(acc_test_1))

# Exibindo, em tela, número ótimo de neurônios ocultos:
print("Número ótimo de neurônios ocultos:", (acc_test_1.index(max(acc_test_1)))+1)

### MÉTODO 2 - RIXM
Modelo SFNN considerando os pesos iniciais e bias do neurônio oculto e do neurônio de saída obtidos a partir do método de Xavier, desconsiderando os pesos dos neurônios já aprendidos.

### Criando Método 2:

In [ ]:
class Method2:

    def __init__(self, x, y, n):
        # vetor de saídas preditas. Inicialmente um vetor de zeros.
        self.Yh = np.zeros((1, y.shape[0]))
        self.n = n  # número de neurônios ocultos.
        # número de neurônios em cada camada (entrada, intermediária e saída).
        self.dims = [x.shape[0], n, 1]
        # um dicionário que conterá os parâmetros W e b entre as camadas da rede (entrada-intermediária e intermediária-saída); ou seja: W1, b1, W2 e b2.
        self.param = {}
        self.ch = {}  # parâmetro intermediário de cálculo.
        self.grad = {}  # parâmetro intermediário de cálculo.
        self.loss = None  # erro de treinamento (BCE).
        self.lr = 0.1  # taxa de aprendizagem.
        # número de observações do conjunto de treinamento.
        self.sam = y.shape[0]

    def nInit(self):
        self.param['W1'] = np.random.uniform(-math.sqrt(6/(self.dims[0] + self.dims[1])), math.sqrt(
            6/(self.dims[0] + self.dims[1])), (self.dims[1], self.dims[0]))
        self.param['b1'] = np.zeros((self.dims[1], 1))
        self.param['W2'] = np.random.uniform(-math.sqrt(6/(self.dims[1] + self.dims[2])), math.sqrt(
            6/(self.dims[1] + self.dims[2])), (self.dims[2], self.dims[1]))
        self.param['b2'] = np.zeros((self.dims[2], 1))
        return

    def forward(self, x):
        Z1 = self.param['W1'].dot(x) + self.param['b1']
        A1 = sigmoid(Z1)
        self.ch['Z1'], self.ch['A1'] = Z1, A1
        Z2 = self.param['W2'].dot(A1) + self.param['b2']
        A2 = sigmoid(Z2)
        self.ch['Z2'], self.ch['A2'] = Z2, A2
        self.Yh = A2
        return self.Yh

    def backward(self, x, y):
        dLoss_Yh = - (y-self.Yh[0])

        dLoss_Z2 = dLoss_Yh * dsigmoid(self.ch['Z2'])
        dLoss_A1 = np.dot(self.param["W2"].T, dLoss_Z2)
        dLoss_W2 = 1/self.ch['A1'].shape[1] * np.dot(dLoss_Z2, self.ch['A1'].T)
        dLoss_b2 = 1/self.ch['A1'].shape[1] * \
            np.dot(dLoss_Z2, np.ones([dLoss_Z2.shape[1], 1]))

        dLoss_Z1 = dLoss_A1 * dsigmoid(self.ch['Z1'])
        dLoss_A0 = np.dot(self.param["W1"].T, dLoss_Z1)
        dLoss_W1 = 1/x.shape[1] * np.dot(dLoss_Z1, x.T)
        dLoss_b1 = 1/x.shape[1] * \
            np.dot(dLoss_Z1, np.ones([dLoss_Z1.shape[1], 1]))

        self.param["W1"] = self.param["W1"] - self.lr * dLoss_W1
        self.param["b1"] = self.param["b1"] - self.lr * dLoss_b1
        self.param["W2"] = self.param["W2"] - self.lr * dLoss_W2
        self.param["b2"] = self.param["b2"] - self.lr * dLoss_b2

    def train(self, x, y, iter=1501):
        for i in range(0, iter):
            self.forward(x)
            self.backward(x, y)
        self.loss = bce(y, self.Yh[0])
        return

    def test(self, x, y):
        self.forward(x)
        return bce(y, self.Yh[0])

### Executando Método 2:

In [ ]:
# Inicializando contagem do tempo:
ini = time.time()

# Abrindo listas para BCE e Acurácia (treino e teste):
bce_train_2 = []
bce_test_2 = []
acc_train_2 = []
acc_test_2 = []

# Inicializando loop para neurônios ocultos:
for n in range(1, 251):
    # Sinalização em tela de início de rede com n neurônios ocultos:
    print("Iniciando rede com", n, "neurônios ocultos:")

    # Definindo entrada e saída de treino:
    x = train_inputs.T
    y = train_targets.T

    # Criando modelo com n neurônios ocultos:
    Modelo = Method2(x, y, n)

    # Inicializando pesos:
    Modelo.nInit()

    # Treinando modelo:
    Modelo.train(x, y)

    # Registrando e exibindo BCE de treino:
    bce_train_2.append(Modelo.loss)
    print("BCE de treino:", Modelo.loss)

    # Registrando e exibindo Acurácia de treino:
    for n in range(0, Modelo.Yh.shape[1]):
        if Modelo.Yh[0][n] > 0.5:
            Modelo.Yh[0][n] = 1
        else:
            Modelo.Yh[0][n] = 0
    acc_train_2.append(accuracy_score(y, Modelo.Yh[0]))
    print("Acurácia de treino:", accuracy_score(y, Modelo.Yh[0]))

    # Definindo entrada e saída de teste:
    x = test_inputs.T
    y = test_targets.T

    # Testando modelo e registrando BCE de teste:
    bce_test_2.append(Modelo.test(x, y))

    # Registrando e exibindo Acurácia de teste:
    for n in range(0, Modelo.Yh.shape[1]):
        if Modelo.Yh[0][n] > 0.5:
            Modelo.Yh[0][n] = 1
        else:
            Modelo.Yh[0][n] = 0
    acc_test_2.append(accuracy_score(y, Modelo.Yh[0]))
    print("Acurácia de teste:", accuracy_score(y, Modelo.Yh[0]))

### ANÁLISE DA PERFORMANCE (MÉTRICAS: FUNÇÃO DE PERDA [BCE DE TREINO], ACURÁCIA DE TREINO, BCE DE TESTE E ACURÁCIA DE TESTE)

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento:", fim-ini)

In [ ]:
# Criação e exibição de Gráfico do BCE de treino:
plt.plot(bce_train_2)
plt.ylabel('Train BCE')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico da Acurácia de treino:
plt.plot(acc_train_2)
plt.ylabel('Train Accuracy')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

# Exibindo, em tela, Acurácia de treino associada ao modelo com maior Acurácia de teste:
print("Acurácia de treino associada ao melhor modelo obtido:",
      acc_train_2[acc_test_2.index(max(acc_test_2))])

In [ ]:
# Criação e exibição de Gráfico do BCE de treino:
plt.plot(bce_test_2)
plt.ylabel('Test BCE')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Gráfico do Acurácia de teste:
plt.plot(acc_test_2)
plt.ylabel('Test Accuracy')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

# Exibindo, em tela, Acurácia máximo de teste:
print("Acurácia máxima de teste obtida:", max(acc_test_2))

# Exibindo, em tela, número ótimo de neurônios ocultos:
print("Número ótimo de neurônios ocultos:", (acc_test_2.index(max(acc_test_2)))+1)

### MÉTODO 3 - ELM
Modelo ELM considerando os pesos iniciais e bias do neurônio oculto e do neurônio de saída obtidos a partir do método de inicialização randômica clássica, desconsiderando os pesos já aprendidos.

### Definindo função de ativação ReLU:

In [ ]:
def relu(x):
    return np.maximum(x, 0, x)

### Criando Método 3:

In [ ]:
def hidden_nodes(X):
    G = np.dot(X, input_weights)
    G = G + biases
    H = relu(G)
    return H


def predict_test(X):
    out = hidden_nodes(X)
    out = np.dot(out, output_weights)
    return out

### Executando Método 3:

In [ ]:
# Inicializando contagem de tempo:
ini = time.time()

# Abrindo listas para Acurácia (treino e teste):
bce_train_3 = []
acc_train_3 = []
bce_test_3 = []
acc_test_3 = []

# Definindo tamanho do conjunto de entrada:
input_size = train_inputs.shape[1]

# Inicializando loop para neurônios ocultos:
for n in range(1, 251):
    # Treino:
    input_weights = np.random.uniform(-1, 1, (input_size, n))
    biases = np.zeros(n)
    print("Iniciando rede com", n, "neurônios ocultos")
    out = hidden_nodes(train_inputs)
    output_weights = np.dot(pinv(out), train_targets)
    out = np.dot(out, output_weights)
    actual = train_targets
    bce_train_3.append(bce(actual, out))
    for n in range(0, out.shape[0]):
        if out[n] > 0.5:
            out[n] = 1
        else:
            out[n] = 0
    acc_train_3.append(accuracy_score(actual, out))

    # Teste:
    test_predict = predict_test(test_inputs)
    for n in range(0, test_predict.shape[0]):
        if test_predict[n] > 0.5:
            test_predict[n] = 1
        else:
            test_predict[n] = 0
    actual = test_targets
    bce_test_3.append(bce(actual, test_predict))
    acc_test_3.append(accuracy_score(actual, test_predict))

### ANÁLISE DA PERFORMANCE (MÉTRICAS: FUNÇÃO DE PERDA [BCE DE TREINO], ACURÁCIA DE TREINO, BCE DE TESTE E ACURÁCIA DE TESTE)

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento:", fim-ini)

In [ ]:
# Criação e exibição de Gráfico do BCE de treino:
plt.plot(bce_train_3)
plt.ylabel('Train BCE')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico da Acurácia de treino:
plt.plot(acc_train_3)
plt.ylabel('Train Accuracy')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()
print("Acurácia de treino associada ao melhor modelo obtido:",
      acc_train_3[acc_test_3.index(max(acc_test_3))])

In [ ]:
# Criação e exibição de Gráfico do BCE de teste:
plt.plot(bce_test_3)
plt.ylabel('Test BCE')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico da Acurácia de teste:
plt.plot(acc_test_3)
plt.ylabel('Test Accuracy')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()
print("Acurácia máximo de teste obtido:", max(acc_test_3))
print("Número ótimo de neurônios ocultos:", (acc_test_3.index(max(acc_test_3)))+1)

## GRÁFICOS DOS TRÊS MÉTODOS EM CONJUNTO

In [ ]:
# Gráfico em conjunto do BCE de treino (até 250):
x = np.arange(1, 251)
y1 = bce_train_1
y2 = bce_train_2
y3 = bce_train_3

# Initialise the figure and axes.
fig, ax = plt.subplots(1, figsize=(8, 6))

# Draw all the lines in the same plot, assigning a label for each one to be shown in the legend.
ax.plot(x, y1, color="red", label="WILCA-BC")
ax.plot(x, y2, color="green", label="RIXM")
ax.plot(x, y3, color="blue", label="ELM")

# Add a legend
plt.legend()

# Nome dos eixos:
plt.ylabel('Train BCE')
plt.xlabel('Number of hidden units')
# TO ZOOM THE GRAPH:
# plt.xlim(0,50)
plt.show()

In [ ]:
# Gráfico em conjunto do Acurácia de teste (até 250):
x = np.arange(1, 251)
acc_test1_100 = [x * 100 for x in acc_test_1]
acc_test2_100 = [x * 100 for x in acc_test_2]
acc_test3_100 = [x * 100 for x in acc_test_3]
y1 = acc_test1_100
y2 = acc_test2_100
y3 = acc_test3_100

# Initialise the figure and axes.
fig, ax = plt.subplots(1, figsize=(8, 6))

# Draw all the lines in the same plot, assigning a label for each one to be shown in the legend.
ax.plot(x, y1, color="red", label="WILCA-BC")
ax.plot(x, y2, color="green", label="RIXM")
ax.plot(x, y3, color="blue", label="ELM")

# Add a legend
plt.legend()

# Nome dos eixos:
plt.ylabel('Test accuracy')
plt.xlabel('Number of hidden units')
# TO ZOOM THE GRAPH:
# plt.xlim(0,50)
plt.show()

### FIM DO CÓDIGO